# Notebook 3: Stationarity Testing + Ornstein-Uhlenbeck Fit

Validates that the HYSYS-weighted partial product spread is mean-reverting,
then fits an Ornstein-Uhlenbeck process to extract the parameters that drive
the trading signal.

## Train / test split

All parameter selection (stationarity validation, OU fit, z-score baseline,
entry threshold) is performed on the **training set only**. The **test set is
never touched** in this notebook — it is held out entirely for out-of-sample
backtesting in Notebook 4.

```
|------------------- Train (70%) -------------------|------- Test (30%) -------|
        fit OU params, validate stationarity              held out completely
        choose entry threshold                            used only in NB4
```

## Tests performed
1. **ADF (Augmented Dickey-Fuller)** — rejects unit root (H0: non-stationary)
2. **KPSS** — confirms stationarity (H0: stationary)
3. **OU parameter estimation** — Maximum Likelihood Estimation

## OU Process
```
dX(t) = theta*(mu - X(t))dt + sigma*dW(t)
```
- **theta:** mean reversion speed
- **mu:** long-run equilibrium spread
- **sigma:** volatility
- **Half-life:** ln(2)/theta — practical trading horizon in days

---

## Design decisions

**Lookback of 126 days, not 252.** A one-year rolling window consumes a full year
of the sample as warm-up before the first signal can fire. Six months preserves
more usable data and makes the z-score more responsive to regime changes.

**Exit at ±0.5σ, not at 0.** With a mean-reversion half-life of roughly 20 days,
waiting for full reversion to the mean holds positions for months and generates
almost no trades. Exiting partway back captures most of the expected move while
producing a trade count large enough to evaluate.

**Minimum trade count filter on threshold selection.** A threshold that produces
very few trades can post a flattering Sharpe by luck. Thresholds generating fewer
than 20 training trades are excluded from selection.

**Parameters saved as JSON, not CSV.** A `pd.Series` mixing floats with a date
string becomes object dtype, and values were corrupted on the CSV round-trip
(`entry_threshold` returned as a bool). JSON preserves types.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from statsmodels.tsa.stattools import adfuller, kpss
from signal_generator import fit_ou_process, generate_signals, apply_holding_period
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# PARAMETERS
# ============================================================

LOOKBACK         = 126     # 6-month rolling window for the z-score
EXIT_THRESHOLD   = 0.5     # exit at +/-0.5 sigma, not full reversion to 0
MIN_HOLDING_DAYS = 5       # operational inertia (HYSYS-derived)
TRANSACTION_COST = 0.05    # $/bbl per trade
MIN_TRAIN_TRADES = 20      # reject thresholds too inactive to evaluate

print(f'Lookback:          {LOOKBACK} days')
print(f'Exit threshold:    +/-{EXIT_THRESHOLD} sigma')
print(f'Min holding:       {MIN_HOLDING_DAYS} days')
print(f'Transaction cost:  ${TRANSACTION_COST}/bbl per trade')
print(f'Min train trades:  {MIN_TRAIN_TRADES}')

In [ ]:
# Load spread data
df = pd.read_csv('../data/spread_data.csv', index_col=0, parse_dates=True)
df = df.dropna(subset=['spread_hysys'])
print(f'Total series length: {len(df)} observations')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT - chronological, 70/30
# Test set is held out completely from this point forward.
# ============================================================

split_idx  = int(len(df) * 0.70)
split_date = df.index[split_idx]

train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

print(f'Train set: {len(train)} obs  ({train.index[0].date()} to {train.index[-1].date()})')
print(f'Test set:  {len(test)} obs  ({test.index[0].date()} to {test.index[-1].date()})')
print(f'Split date: {split_date.date()}')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train.index, train['spread_hysys'], color='#2c3e50', linewidth=0.8, label='Train (in-sample)')
ax.plot(test.index,  test['spread_hysys'],  color='#e67e22', linewidth=0.8, label='Test (held out)')
ax.axvline(split_date, color='red', linestyle='--', linewidth=1, label='Train/Test split')
ax.set_ylabel('Spread ($/bbl)')
ax.set_title('Train/Test Split - HYSYS Partial Product Spread')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# STATIONARITY TESTS - TRAIN SET ONLY
# ============================================================

train_spread = train['spread_hysys']

adf_stat, adf_pvalue, adf_lags, adf_nobs, adf_crit, _ = adfuller(train_spread, autolag='AIC')

print('=' * 55)
print('AUGMENTED DICKEY-FULLER TEST (TRAIN SET)')
print('H0: Unit root exists (series is NON-stationary)')
print('=' * 55)
print(f'ADF Statistic:  {adf_stat:.4f}')
print(f'p-value:        {adf_pvalue:.4f}')
print(f'Lags used:      {adf_lags}')
print('Critical values:')
for key, val in adf_crit.items():
    print(f'  {key}: {val:.4f}')
print()
if adf_pvalue < 0.05:
    print('RESULT: Reject H0 at 5% significance -> series IS stationary (train set)')
else:
    print('RESULT: Fail to reject H0 -> series may NOT be stationary (train set)')

In [ ]:
# KPSS Test - TRAIN SET ONLY - H0: series IS stationary
kpss_stat, kpss_pvalue, kpss_lags, kpss_crit = kpss(train_spread, regression='c', nlags='auto')

print('=' * 55)
print('KPSS TEST (TRAIN SET)')
print('H0: Series IS stationary')
print('=' * 55)
print(f'KPSS Statistic: {kpss_stat:.4f}')
print(f'p-value:        {kpss_pvalue:.4f}')
print('Critical values:')
for key, val in kpss_crit.items():
    print(f'  {key}: {val:.4f}')
print()
if kpss_pvalue > 0.05:
    print('RESULT: Fail to reject H0 -> series IS stationary (train set)')
else:
    print('RESULT: Reject H0 -> series may NOT be stationary (train set)')
print()
print('Both tests agreeing (ADF rejects, KPSS fails to reject) is stronger evidence')
print('than either alone, since ADF has low power against near-unit-root alternatives.')

In [ ]:
# ============================================================
# OU PARAMETER ESTIMATION (MLE) - TRAIN SET ONLY
# These parameters are fixed after this cell and applied
# unchanged to the test set in Notebook 4.
# ============================================================

ou_params = fit_ou_process(train_spread)
theta     = ou_params['theta']
mu        = ou_params['mu']
sigma     = ou_params['sigma']
half_life = ou_params['half_life']
stat_sd   = sigma / np.sqrt(2 * theta)

print('=' * 55)
print('ORNSTEIN-UHLENBECK PARAMETERS (MLE, TRAIN SET)')
print('dX = theta(mu - X)dt + sigma*dW')
print('=' * 55)
print(f'theta (mean reversion speed): {theta:.4f} per day')
print(f'mu (long-run mean):           ${mu:.4f}/bbl')
print(f'sigma (volatility):           {sigma:.4f}')
print(f'Half-life:                    {half_life:.1f} trading days')
print(f'Stationary sd (sigma/sqrt(2*theta)): ${stat_sd:.3f}/bbl')
print(f'Optimisation converged:       {ou_params["converged"]}')
print()
print('Tradeability check:')
print(f'  Expected move over one half-life from a 1.75-sigma entry:')
print(f'    ~${0.5 * 1.75 * stat_sd:.2f}/bbl')
print(f'  Noise accumulated over the same {half_life:.0f} days:')
print(f'    sigma*sqrt(t) = ${sigma * np.sqrt(half_life):.2f}/bbl')
print('  If noise exceeds the expected move, the reversion is too slow')
print('  relative to volatility for the signal to be reliably tradeable.')

In [ ]:
# ============================================================
# Z-SCORE - TRAIN SET
# The rolling window is a mechanical, strictly backward-looking
# calculation, not a fitted parameter - at any point t it uses
# only data up to and including t, so applying the same window
# to test data in NB4 does not constitute look-ahead bias.
# ============================================================

train['margin_mean'] = train_spread.rolling(LOOKBACK).mean()
train['margin_std']  = train_spread.rolling(LOOKBACK).std()
train['zscore']      = (train_spread - train['margin_mean']) / train['margin_std']

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

axes[0].plot(train.index, train_spread, color='#2c3e50', linewidth=0.8, alpha=0.8,
             label='HYSYS spread (train)')
axes[0].plot(train.index, train['margin_mean'], color='#e74c3c', linewidth=1.5,
             label=f'Rolling mean ({LOOKBACK}d)')
axes[0].axhline(mu, color='#27ae60', linestyle='--', linewidth=1.2,
                label=f'OU mu = ${mu:.2f}/bbl')
axes[0].fill_between(train.index,
    train['margin_mean'] - train['margin_std'],
    train['margin_mean'] + train['margin_std'],
    alpha=0.15, color='#e74c3c', label='+/-1 sigma band')
axes[0].set_ylabel('Spread ($/bbl)')
axes[0].set_title(f'Train Set - HYSYS Spread - OU Half-life: {half_life:.0f} trading days')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].plot(train.index, train['zscore'], color='#2c3e50', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].axhline(EXIT_THRESHOLD,  color='#3498db', linewidth=1, linestyle='--',
                label=f'+/-{EXIT_THRESHOLD} sigma exit')
axes[1].axhline(-EXIT_THRESHOLD, color='#3498db', linewidth=1, linestyle='--')
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/stationarity_ou_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# THRESHOLD SELECTION - TRAIN SET ONLY
# The sweep uses the same exit rule that will be traded in NB4,
# and rejects thresholds with too few trades to be meaningful.
# ============================================================

thresholds = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5]
results = []

margin_change_train = train_spread.diff()

for thresh in thresholds:
    sig  = generate_signals(train['zscore'], thresh, EXIT_THRESHOLD)
    sig  = apply_holding_period(sig, MIN_HOLDING_DAYS)
    pnl  = sig.shift(1) * margin_change_train - sig.diff().abs().clip(0,1) * TRANSACTION_COST
    sh   = (pnl.mean() / pnl.std()) * np.sqrt(252) if pnl.std() > 0 else 0
    ntrd = sig.diff().abs().clip(0,1).sum() / 2
    results.append({'threshold': thresh, 'sharpe': sh,
                    'num_trades': ntrd, 'total_pnl': pnl.sum()})

sensitivity = pd.DataFrame(results)
print('Threshold sensitivity (TRAIN SET ONLY):')
print(sensitivity.round(3).to_string(index=False))

viable = sensitivity[sensitivity['num_trades'] >= MIN_TRAIN_TRADES]
if len(viable) == 0:
    print(f'\n*** WARNING: no threshold produced >= {MIN_TRAIN_TRADES} train trades. ***')
    print('*** Sample too inactive to select on - extend the data range in NB1. ***')
    viable = sensitivity

best_threshold = float(viable.loc[viable['sharpe'].idxmax(), 'threshold'])
best_trades    = int(viable.loc[viable['sharpe'].idxmax(), 'num_trades'])
best_sharpe    = float(viable.loc[viable['sharpe'].idxmax(), 'sharpe'])

print(f'\nSelected entry threshold: {best_threshold} sigma')
print(f'  train trades: {best_trades}')
print(f'  train Sharpe: {best_sharpe:.3f}  (in-sample, selected on this data -')
print(f'                 treat as an optimistic upper bound)')
print('This threshold is now FIXED and applied to the held-out test set in NB4.')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sensitivity['threshold'], sensitivity['sharpe'], 'o-', color='#2c3e50',
        linewidth=2, label='Train Sharpe')
ax.axvline(best_threshold, color='#e74c3c', linestyle='--',
           label=f'Selected: {best_threshold} sigma')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Entry Threshold (sigma)')
ax.set_ylabel('Train-set Sharpe Ratio')
ax.set_title('Threshold Selection - Train Set Only')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

ax2 = ax.twinx()
ax2.bar(sensitivity['threshold'], sensitivity['num_trades'],
        width=0.08, alpha=0.25, color='#7f8c8d')
ax2.set_ylabel('Number of trades', color='#7f8c8d')
ax2.axhline(MIN_TRAIN_TRADES, color='#7f8c8d', linestyle=':', linewidth=1)

plt.tight_layout()
plt.savefig('../data/threshold_selection_train.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SAVE - JSON, not CSV
# A pd.Series mixing floats with a date string becomes object
# dtype and values are corrupted on the CSV round-trip (this is
# what previously returned entry_threshold as a bool). JSON
# preserves types, and the printout below verifies them.
# ============================================================

params_to_save = {
    'theta':            float(theta),
    'mu':               float(mu),
    'sigma':            float(sigma),
    'half_life':        float(half_life),
    'lookback':         int(LOOKBACK),
    'entry_threshold':  float(best_threshold),
    'exit_threshold':   float(EXIT_THRESHOLD),
    'min_holding_days': int(MIN_HOLDING_DAYS),
    'transaction_cost': float(TRANSACTION_COST),
    'split_date':       str(split_date.date())
}

with open('../data/ou_parameters.json', 'w') as f:
    json.dump(params_to_save, f, indent=2)

df['is_train'] = df.index < split_date
df.to_csv('../data/spread_full_with_split.csv')

print('Saved parameters to ../data/ou_parameters.json:')
for k, v in params_to_save.items():
    print(f'  {k:18s} {v!r:>14}  ({type(v).__name__})')
print()
print(f'Saved {len(df)} rows with train/test flag to ../data/spread_full_with_split.csv')